<a href="https://colab.research.google.com/github/jinbaaaaaang/huwari/blob/main/huwari_harmony_retrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

SAVE_DIR = "/content/drive/MyDrive/fashion_harmony"

# 기존 모델 백업
shutil.copy(
    f"{SAVE_DIR}/fashion_harmony_final.pt",
    f"{SAVE_DIR}/fashion_harmony_final_backup.pt"
)
print("백업 완료!")

Mounted at /content/drive
백업 완료!


## 설치

In [ ]:
!pip install timm datasets --quiet

## 임포트

In [ ]:
import os
import numpy as np
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"디바이스: {device}")

SAVE_DIR      = "/content/drive/MyDrive/fashion_harmony"
HARMONY_SAVE  = f"{SAVE_DIR}/fashion_harmony_retrained.pt"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
디바이스: cuda


## 클래스 정의 (속성 헤드 학습과 동일하게)

In [ ]:
CATEGORY_CLASSES = ["상의", "하의", "신발", "모자", "악세서리"]

MATERIAL_CLASSES = [
    "데님", "니트", "실크", "가죽", "울", "면", "패딩", "기타"
]

PATTERN_CLASSES = [
    "무지", "스트라이프", "체크", "도트", "플로럴",
    "그래픽", "호피·뱀피", "카무플라쥬", "기타"
]

STYLE_CLASSES = [
    "캐주얼", "고프코어", "미니멀", "긱시크", "로맨틱",
    "빈티지", "포멀", "Y2K", "스트리트", "스포티"
]

print(f"재질: {len(MATERIAL_CLASSES)}개")
print(f"패턴: {len(PATTERN_CLASSES)}개")
print(f"스타일: {len(STYLE_CLASSES)}개")

재질: 8개
패턴: 9개
스타일: 10개


## 모델 정의

In [ ]:
class FashionBackbone(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        base            = timm.create_model("efficientnet_b3", pretrained=False)
        self.features   = nn.Sequential(*list(base.children())[:-1])
        in_features     = base.classifier.in_features
        self.projection = nn.Sequential(
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embed_dim),
        )

    def forward(self, x):
        feat = self.features(x).flatten(1)
        return self.projection(feat)


class AttributeHeads(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()

        def _head(out_dim):
            return nn.Sequential(
                nn.Linear(embed_dim, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, out_dim)
            )

        self.category_head = _head(len(CATEGORY_CLASSES))
        self.material_head = _head(len(MATERIAL_CLASSES))
        self.pattern_head  = _head(len(PATTERN_CLASSES))
        self.style_head    = _head(len(STYLE_CLASSES))

    def forward(self, emb):
        return {
            "category": self.category_head(emb),
            "material": self.material_head(emb),
            "pattern":  self.pattern_head(emb),
            "style":    self.style_head(emb),
        }


class SetTransformer(nn.Module):
    def __init__(self, input_dim=1024, nhead=8, num_layers=2):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim, nhead=nhead,
            dim_feedforward=1024, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, input_dim))
        self.score_head  = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x, mask=None):
        B   = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)

        if mask is not None:
            cls_mask         = torch.ones(B, 1, device=mask.device)
            full_mask        = torch.cat([cls_mask, mask], dim=1)
            key_padding_mask = (full_mask == 0)
        else:
            key_padding_mask = None

        out   = self.transformer(x, src_key_padding_mask=key_padding_mask)
        score = self.score_head(out[:, 0, :]).squeeze(1)
        return score


class FashionHarmonyModel(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.backbone        = FashionBackbone(embed_dim)
        self.attr_heads      = AttributeHeads(embed_dim)
        attr_dim             = (len(CATEGORY_CLASSES) + len(MATERIAL_CLASSES) +
                                len(PATTERN_CLASSES)  + len(STYLE_CLASSES))
        self.attr_proj       = nn.Linear(attr_dim, embed_dim)
        self.set_transformer = SetTransformer(input_dim=embed_dim * 2)

    def forward(self, outfit_imgs, mask=None):
        B, N, C, H, W = outfit_imgs.shape
        flat     = outfit_imgs.view(B * N, C, H, W)
        emb      = self.backbone(flat)
        preds    = self.attr_heads(emb)
        attr_vec = torch.cat([
            F.softmax(preds["category"], dim=1),
            F.softmax(preds["material"], dim=1),
            F.softmax(preds["pattern"],  dim=1),
            F.softmax(preds["style"],    dim=1),
        ], dim=1)
        attr_emb = self.attr_proj(attr_vec)
        combined = torch.cat([emb, attr_emb], dim=1).view(B, N, -1)
        score    = self.set_transformer(combined, mask)
        return score

print("모델 정의 완료")

모델 정의 완료


## 모델 로드 (새 속성 헤드 + 기존 Set Transformer)

In [ ]:
model = FashionHarmonyModel(embed_dim=512).to(device)

# 새로 학습한 백본 + 속성 헤드 로드
backbone_ckpt   = f"{SAVE_DIR}/backbone_attr.pt"
attr_heads_ckpt = f"{SAVE_DIR}/attr_heads_kfashion.pt"

model.backbone.load_state_dict(
    torch.load(backbone_ckpt, map_location=device)
)
print("새 백본 로드 완료")

model.attr_heads.load_state_dict(
    torch.load(attr_heads_ckpt, map_location=device)
)
print("새 속성 헤드 로드 완료")

# 기존 Set Transformer 로드 (백업에서)
old_ckpt = torch.load(
    f"{SAVE_DIR}/fashion_harmony_final_backup.pt",
    map_location=device
)
old_state = old_ckpt["model_state_dict"]



print("\n모델 준비 완료!")

새 백본 로드 완료
새 속성 헤드 로드 완료

모델 준비 완료!


## Polyvore-U 데이터셋

In [ ]:
from datasets import load_dataset

print("Polyvore-U 로드 중...")
polyvore = load_dataset("Marqo/polyvore", split="data")
print(f"데이터 수: {len(polyvore)}")

IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                          [0.229, 0.224, 0.225])
])

class OutfitDataset(Dataset):
    def __init__(self, hf_dataset, transform=None, max_items=4):
        self.dataset   = hf_dataset
        self.transform = transform or IMG_TRANSFORM
        self.max_items = max_items

        outfit_to_indices = {}
        for i, row in enumerate(hf_dataset):
            oid = row['item_ID'].rsplit('_', 1)[0]
            if oid not in outfit_to_indices:
                outfit_to_indices[oid] = []
            outfit_to_indices[oid].append(i)

        self.outfit_ids        = [k for k, v in outfit_to_indices.items() if len(v) >= 2]
        self.outfit_to_indices = {k: v for k, v in outfit_to_indices.items() if len(v) >= 2}
        print(f"총 outfit 수: {len(self.outfit_ids)}")

    def __len__(self):
        return len(self.outfit_ids) * 2

    def _load_img(self, row):
        try:
            img = row.get("image") or row.get("img")
            if img is None:
                return Image.new("RGB", (224, 224), (200, 200, 200))
            if isinstance(img, Image.Image):
                return img.convert("RGB")
            return Image.fromarray(np.array(img)).convert("RGB")
        except:
            return Image.new("RGB", (224, 224), (200, 200, 200))

    def __getitem__(self, idx):
        is_positive = np.random.random() > 0.5
        outfit_idx  = idx % len(self.outfit_ids)
        oid         = self.outfit_ids[outfit_idx]
        indices     = self.outfit_to_indices[oid]

        if is_positive:
            selected_indices = indices[:self.max_items]
            label = 1.0
        else:
            neg_oid     = self.outfit_ids[
                (outfit_idx + np.random.randint(1, len(self.outfit_ids) // 2))
                % len(self.outfit_ids)
            ]
            neg_indices  = self.outfit_to_indices[neg_oid]
            mixed        = indices[:self.max_items//2] + neg_indices[:self.max_items//2]
            np.random.shuffle(mixed)
            selected_indices = mixed[:self.max_items]
            label = 0.0

        imgs = [self.transform(self._load_img(self.dataset[i])) for i in selected_indices]
        while len(imgs) < self.max_items:
            imgs.append(torch.zeros(3, 224, 224))

        imgs_tensor = torch.stack(imgs)
        mask        = torch.zeros(self.max_items)
        mask[:len(selected_indices)] = 1.0

        return imgs_tensor, mask, torch.tensor(label, dtype=torch.float32)

outfit_dataset = OutfitDataset(polyvore, max_items=4)
outfit_loader  = DataLoader(outfit_dataset, batch_size=16, shuffle=True,
                             num_workers=2, pin_memory=True)
print("데이터셋 준비 완료")

Polyvore-U 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/data-00000-of-00006.parquet:   0%|          | 0.00/428M [00:00<?, ?B/s]

data/data-00001-of-00006.parquet:   0%|          | 0.00/421M [00:00<?, ?B/s]

...

data/data-00003-of-00006.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

data/data-00004-of-00006.parquet:   0%|          | 0.00/422M [00:00<?, ?B/s]

data/data-00005-of-00006.parquet:   0%|          | 0.00/409M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/94096 [00:00<?, ? examples/s]

데이터 수: 94096
총 outfit 수: 20614
데이터셋 준비 완료


## Set Transformer 재학습

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(),        "lr": 1e-6},
    {"params": model.attr_heads.parameters(),      "lr": 1e-5},
    {"params": model.attr_proj.parameters(),       "lr": 1e-3},
    {"params": model.set_transformer.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

harmony_loss_fn = nn.BCELoss()

EPOCHS      = 20
ACCUMULATION = 4  # 배치 16 × 4 = 배치 64 효과
best_acc    = 0

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0
    correct    = 0
    total      = 0

    optimizer.zero_grad()
    pbar = tqdm(outfit_loader, desc=f"ST {epoch+1}/{EPOCHS}")

    for i, (imgs, mask, label) in enumerate(pbar):
        imgs  = imgs.to(device)
        mask  = mask.to(device)
        label = label.to(device)

        score = model(imgs, mask)
        loss  = harmony_loss_fn(score, label) / ACCUMULATION
        loss.backward()

        if (i + 1) % ACCUMULATION == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        pred     = (score > 0.5).float()
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * ACCUMULATION

        pbar.set_postfix({
            "loss": f"{loss.item()*ACCUMULATION:.4f}",
            "acc":  f"{correct/total:.3f}"
        })

    scheduler.step()
    acc = correct / total
    print(f"Epoch {epoch+1}: loss={total_loss/len(outfit_loader):.4f}, acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch":            epoch,
            "acc":              acc,
        }, HARMONY_SAVE)
        print(f"  → 저장 완료 (acc={acc:.4f})")

print(f"\n학습 완료! 최고 acc={best_acc:.4f}")

ST 1/20:   0%|          | 0/2577 [00:00<?, ?it/s]

Epoch 1: loss=0.5848, acc=0.6863
  → 저장 완료 (acc=0.6863)


ST 2/20:   0%|          | 0/2577 [00:00<?, ?it/s]

Epoch 2: loss=0.5545, acc=0.7086
  → 저장 완료 (acc=0.7086)


ST 3/20:   0%|          | 0/2577 [00:00<?, ?it/s]

...

Epoch 18: loss=0.4771, acc=0.7627


ST 19/20:   0%|          | 0/2577 [00:00<?, ?it/s]

Epoch 19: loss=0.4731, acc=0.7663
  → 저장 완료 (acc=0.7663)


ST 20/20:   0%|          | 0/2577 [00:00<?, ?it/s]

Epoch 20: loss=0.4703, acc=0.7683
  → 저장 완료 (acc=0.7683)

학습 완료! 최고 acc=0.7683


## AUC 평가

In [ ]:
from sklearn.metrics import roc_auc_score

model.eval()
all_scores = []
all_labels = []

eval_dataset = OutfitDataset(polyvore, max_items=4)
eval_loader  = DataLoader(eval_dataset, batch_size=16, shuffle=False,
                           num_workers=2, pin_memory=True)

print("평가 중...")
with torch.no_grad():
    for imgs, mask, label in tqdm(eval_loader):
        imgs  = imgs.to(device)
        mask  = mask.to(device)
        score = model(imgs, mask)
        all_scores.extend(score.cpu().numpy())
        all_labels.extend(label.numpy())

auc = roc_auc_score(all_labels, all_scores)
acc = sum(1 for s, l in zip(all_scores, all_labels)
          if (s > 0.5) == (l > 0.5)) / len(all_labels)

print(f"\n===== 평가 결과 =====")
print(f"AUC:      {auc:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"기존 AUC: 0.9120")
print(f"차이:     {auc - 0.9120:+.4f}")

총 outfit 수: 20614
평가 중...


  0%|          | 0/2577 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



===== 평가 결과 =====
AUC:      0.8708
Accuracy: 0.7815
기존 AUC: 0.9120
차이:     -0.0412


In [ ]:
import os
SAVE_DIR = "/content/drive/MyDrive/fashion_harmony"
print(os.listdir(SAVE_DIR))

['backbone_cl.pt', 'auto_labels.json', 'auto_labels_balanced.json', 'set_transformer.pt', 'model_st.pt', 'attr_proj.pt', 'attr_heads.pt', 'fashion_harmony_final.pt', 'attr_heads_kfashion.pt', 'backbone_attr.pt', 'fashion_harmony_final_backup.pt', 'fashion_harmony_retrained.pt']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
files.download("/content/drive/MyDrive/fashion_harmony/fashion_harmony_retrained.pt")

Mounted at /content/drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ================================================================
# 드라이브에서 모델 불러와서 테스트
# ================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "/content/drive/MyDrive/fashion_harmony"

# 클래스 정의
CATEGORY_CLASSES = ["상의", "하의", "신발", "모자", "악세서리"]
MATERIAL_CLASSES = ["데님", "니트", "실크", "가죽", "울", "면", "패딩", "기타"]
PATTERN_CLASSES  = ["무지", "스트라이프", "체크", "도트", "플로럴", "그래픽", "호피·뱀피", "카무플라쥬", "기타"]
STYLE_CLASSES    = ["캐주얼", "고프코어", "미니멀", "긱시크", "로맨틱", "빈티지", "포멀", "Y2K", "스트리트", "스포티"]

# 모델 정의
class FashionBackbone(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        base            = timm.create_model("efficientnet_b3", pretrained=False)
        self.features   = nn.Sequential(*list(base.children())[:-1])
        self.projection = nn.Sequential(
            nn.Linear(base.classifier.in_features, 1024),
            nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(1024, embed_dim),
        )
    def forward(self, x):
        return self.projection(self.features(x).flatten(1))

class AttributeHeads(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        def _head(n):
            return nn.Sequential(
                nn.Linear(embed_dim, 256), nn.BatchNorm1d(256),
                nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n)
            )
        self.category_head = _head(len(CATEGORY_CLASSES))
        self.material_head = _head(len(MATERIAL_CLASSES))
        self.pattern_head  = _head(len(PATTERN_CLASSES))
        self.style_head    = _head(len(STYLE_CLASSES))
    def forward(self, emb):
        return {
            "category": self.category_head(emb),
            "material": self.material_head(emb),
            "pattern":  self.pattern_head(emb),
            "style":    self.style_head(emb),
        }

class SetTransformer(nn.Module):
    def __init__(self, input_dim=1024, nhead=8, num_layers=2):
        super().__init__()
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=input_dim, nhead=nhead,
            dim_feedforward=1024, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, input_dim))
        self.score_head  = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x, mask=None):
        B   = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        if mask is not None:
            cls_mask = torch.ones(B, 1, device=mask.device)
            full_mask = torch.cat([cls_mask, mask], dim=1)
            key_padding_mask = (full_mask == 0)
        else:
            key_padding_mask = None
        out = self.transformer(x, src_key_padding_mask=key_padding_mask)
        return self.score_head(out[:, 0, :]).squeeze(1)

class FashionHarmonyModel(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.backbone        = FashionBackbone(embed_dim)
        self.attr_heads      = AttributeHeads(embed_dim)
        attr_dim             = len(CATEGORY_CLASSES) + len(MATERIAL_CLASSES) + len(PATTERN_CLASSES) + len(STYLE_CLASSES)
        self.attr_proj       = nn.Linear(attr_dim, embed_dim)
        self.set_transformer = SetTransformer(input_dim=embed_dim * 2)
    def forward(self, outfit_imgs, mask=None):
        B, N, C, H, W = outfit_imgs.shape
        flat     = outfit_imgs.view(B * N, C, H, W)
        emb      = self.backbone(flat)
        preds    = self.attr_heads(emb)
        attr_vec = torch.cat([
            F.softmax(preds["category"], dim=1),
            F.softmax(preds["material"], dim=1),
            F.softmax(preds["pattern"],  dim=1),
            F.softmax(preds["style"],    dim=1),
        ], dim=1)
        attr_emb = self.attr_proj(attr_vec)
        combined = torch.cat([emb, attr_emb], dim=1).view(B, N, -1)
        return self.set_transformer(combined, mask)

# 모델 로드
model = FashionHarmonyModel().to(device)
ckpt  = torch.load(f"{SAVE_DIR}/fashion_harmony_retrained.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"모델 로드 완료 (학습 acc: {ckpt['acc']:.4f})")

# Polyvore-U로 테스트
from datasets import load_dataset
import torchvision.transforms as transforms

polyvore = load_dataset("Marqo/polyvore", split="data")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# outfit 그룹핑
outfit_to_indices = {}
for i, row in enumerate(polyvore):
    oid = row['item_ID'].rsplit('_', 1)[0]
    if oid not in outfit_to_indices:
        outfit_to_indices[oid] = []
    outfit_to_indices[oid].append(i)

outfit_ids = [k for k, v in outfit_to_indices.items() if len(v) >= 2]

scores_pos = []
scores_neg = []

with torch.no_grad():
    for i in range(200):
        oid     = outfit_ids[i]
        indices = outfit_to_indices[oid]

        # positive
        imgs = []
        for idx in indices[:4]:
            row = polyvore[idx]
            img = row["image"].convert("RGB")
            imgs.append(transform(img))
        while len(imgs) < 4:
            imgs.append(torch.zeros(3, 224, 224))
        imgs_t = torch.stack(imgs).unsqueeze(0).to(device)
        mask   = torch.zeros(1, 4).to(device)
        mask[0, :len(indices[:4])] = 1.0
        score  = model(imgs_t, mask).item()
        scores_pos.append(score)

        # negative (랜덤 조합)
        neg_oid = outfit_ids[(i + 100) % len(outfit_ids)]
        neg_idx = outfit_to_indices[neg_oid]
        mixed   = indices[:2] + neg_idx[:2]
        imgs = []
        for idx in mixed:
            row = polyvore[idx]
            img = row["image"].convert("RGB")
            imgs.append(transform(img))
        imgs_t = torch.stack(imgs).unsqueeze(0).to(device)
        mask   = torch.ones(1, 4).to(device)
        score  = model(imgs_t, mask).item()
        scores_neg.append(score)

print(f"\npositive 평균: {sum(scores_pos)/len(scores_pos):.4f}")
print(f"negative 평균: {sum(scores_neg)/len(scores_neg):.4f}")
print(f"positive 범위: {min(scores_pos):.4f} ~ {max(scores_pos):.4f}")
print(f"negative 범위: {min(scores_neg):.4f} ~ {max(scores_neg):.4f}")

Mounted at /content/drive
모델 로드 완료 (학습 acc: 0.7683)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/data-00000-of-00006.parquet:   0%|          | 0.00/428M [00:00<?, ?B/s]

data/data-00001-of-00006.parquet:   0%|          | 0.00/421M [00:00<?, ?B/s]

...

data/data-00004-of-00006.parquet:   0%|          | 0.00/422M [00:00<?, ?B/s]

data/data-00005-of-00006.parquet:   0%|          | 0.00/409M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/94096 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



positive 평균: 0.6965
negative 평균: 0.3046
positive 범위: 0.1085 ~ 0.9985
negative 범위: 0.0101 ~ 0.9394
